# Study how peakedness depends on the average deviation of the block replacement

In [1]:
import qiskit
from qiskit import QuantumRegister as Q_R
from qiskit import ClassicalRegister as C_R
from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator
from qiskit.visualization import array_to_latex
from qiskit.visualization import circuit_drawer

from IPython.display import display

import numpy as np
import random
import copy
import time

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# SciPy minimizer routine
from scipy.optimize import minimize

PI = np.pi

### Definig functions that create a single brick in the brick wall ciruit, create peaked circuits

In [2]:
#Creates a single "brick" block and add it to an existing circuit
def add_double_cz_block(qc, params, qubits_list):
    qc.u(params[0], params[1], params[2], qubits_list[0])
    qc.u(params[3], params[4], params[5], qubits_list[1])
    qc.cz(qubits_list[0], qubits_list[1])
    qc.u(params[6], params[7], params[8], qubits_list[0])
    qc.u(params[9], params[10], params[11], qubits_list[1])
    qc.cz(qubits_list[0], qubits_list[1])
    qc.u(params[12], params[13], params[14], qubits_list[0])
    qc.u(params[15], params[16], params[17], qubits_list[1])
    return qc

#Creates a single inverted "brick" block and add it to an existing circuit
def add_double_inverse_cz_block(qc, params, qubits_list):
    pp = params
    qc.u(-pp[12], -pp[14], -pp[13], qubits_list[0])
    qc.u(-pp[15], -pp[17], -pp[16], qubits_list[1])
    qc.cz(qubits_list[0], qubits_list[1])
    qc.u(-pp[6], -pp[8], -pp[7], qubits_list[0])
    qc.u(-pp[9], -pp[11], -pp[10], qubits_list[1])
    qc.cz(qubits_list[0], qubits_list[1])
    qc.u(-pp[0], -pp[2], -pp[1], qubits_list[0])
    qc.u(-pp[3], -pp[5], -pp[4], qubits_list[1])
    return qc

#Creates a single "brick" block operator
def double_cz_op(par):
    q_2 = qiskit.QuantumCircuit(2)
    q_2.u(par[0], par[1], par[2], 0)
    q_2.u(par[3], par[4], par[5], 1)
    q_2.cz(0,1)
    q_2.u(par[6], par[7], par[8], 0)
    q_2.u(par[9], par[10], par[11], 1)
    q_2.cz(0,1)
    q_2.u(par[12], par[13], par[14], 0)
    q_2.u(par[15], par[16], par[17], 1)
    op = Operator(q_2)
    return op

#Calculate difference between two "brick" blocks
def double_q_diff(mat_1, mat_2):
    mm = mat_1 - mat_2
    #print(mm.data)
    numb_el = mm.data.size
    #print(numb_el)
    return np.linalg.norm(mm)/numb_el

# auxialiry function for optimization
def estim_func(params, init_circ):
    test_circ = double_cz_op(params)
    est = double_q_diff(init_circ, test_circ)
    return est

#Take an intial "brick" block and find another "brick" block that performs alsmost the same function as initial block
def find_alt_params(param_init, prec, n_tries):
    min_dev = 10000
    init_circ = double_cz_op(param_init)
    for i in range(n_tries):
        param_0 = np.random.rand(18) * 2 * PI
        res = minimize(
                    estim_func,
                    param_0,
                    args = (init_circ),
                    method = 'COBYLA', 
                    options = {"maxiter":  5000} 
                )
        if res.fun < min_dev:
            min_dev_par = res
            min_dev = res.fun
        if res.fun < prec:
            break

    return min_dev_par


In [3]:
#The function creates a quasi-random peaked circuit
#Parameter:
#n_qubits - number of qubits
#qc_depth - circuit depth
#add_string - defines if a secret string is added to the ciruit or not. If not, then the secret string is |0>^n
#list_of_switched_qubits - secret string
#list_of_shifts - positions of the secter x operators in the circuit (see the algorith description in the paper: ArXiv...)
#prec - precision of the block replacement
#n_tries - how many times the optimizer tries to achieve the precision
#phi_range - range of angles of the U-gates in the circuit
#added_deviation - we can increase the deviation of the replacement block by increasing this parameter
def peaked_circuit_parameters(n_qubits, qc_depth, prec, n_tries, phi_range):
    
    start_time = time.time()
    n_randon_blocks = int(n_qubits / 2 * qc_depth) #total number of blocks in the first half of the circuit
    params = []

    #Part 1. Creating a ciruit with peak at |0>^n state 
    #create a set of random angles for all blocks in the first half of the circuit
    for i in range(n_randon_blocks):
        pp = []
        for j in range(18):
            pp.append(random.random() * phi_range)
            #pp.append(random.random() * 0.5)
        params.append(pp)
    
    #creating the second half of the circuit. The mirror, or inverse circuit
    #Depending on the 'hidden' parameter, blocks modification is either used or not
    mirror_params = []
    mirror_params_init = []
    for i_block in range(n_randon_blocks):
        param_init = params[n_randon_blocks - 1 - i_block]
        res_min = find_alt_params(param_init, prec, n_tries)
        new_params = res_min.x
        mirror_params.append(new_params)
        mirror_params_init.append(param_init)
 
    return params, mirror_params, mirror_params_init  
    

def create_peaked_circuit(params_rand, params_mirror, mirror_params_init, n_qubits, qc_depth):
    q_reg = Q_R(n_qubits)
    c_reg = C_R(n_qubits)
    qc_rand = qiskit.QuantumCircuit(q_reg, c_reg)
    
    #Creating a list of qubit pairs
    block_n = 0
    qubits_pairs_order = []
    for i in range(qc_depth):
        if i % 2 == 0:
            for i_q in range(int(n_qubits / 2)):
                qubits_pairs_order.append([i_q * 2, i_q * 2 + 1])
                block_n = block_n + 1
        if i % 2 == 1:
            qubits_pairs_order.append([0, n_qubits - 1])
            block_n = block_n + 1
            for i_q in range(int(n_qubits / 2) - 1):
                qubits_pairs_order.append([i_q * 2 + 1, i_q * 2 + 2])
                block_n = block_n + 1
    
    #creating first half of the circuit. The random brick-wall circuit
    for i_block in range(block_n):
        qc_rand = add_double_cz_block(qc_rand, params_rand[i_block], qubits_pairs_order[i_block])
    block_dev= []
    for i_block in range(block_n):
        new_params = params_mirror[i_block] + np.random.rand(18) * added_deviation
        #for i in range(18):
        #    new_params[i] = new_params[i] + random.random() * added_deviation
        qc_rand = add_double_inverse_cz_block(qc_rand, new_params, qubits_pairs_order[block_n - 1 - i_block])
        b_1 = double_cz_op(new_params)
        b_2 = double_cz_op(mirror_params_init[i_block])
        block_dev.append(double_q_diff(b_1, b_2))
        
    
    #add measurements of each qubits at the end
    for i in range(n_qubits):
        qc_rand.measure(i,i)
    
    return qc_rand, block_dev

In [4]:
#Create a random circuit 
n_q = 6 #number of qubits
n_l = 4 #circuit depth
prec = 0.005 # modification precision
n_tries = 3 #number of tries to achieve modification precision
phi_range = 2 * PI #range of angles that is used for U-gates in the random peaked quantum circuit (prior to modification)
params_rand, mirror_params, mirror_params_init = peaked_circuit_parameters(n_q, n_l, prec, n_tries, phi_range)


In [5]:
for i in range(10):
    added_deviation = 0.0 + (i)*0.01
    print(f"Added deviation: {added_deviation}")
    
    block_dev = []
    qc, block_dev = create_peaked_circuit(params_rand, mirror_params, mirror_params_init, n_q, n_l)
    #qc.draw('mpl')
    #show mean block deviation
    mean_block_deviation = np.mean(block_dev)
    print(f"Mean block deviation: {mean_block_deviation}")
    
    #simulate created peaked circuit
    simulator = AerSimulator()
    qc_merged_t = transpile(qc, simulator)
    result = simulator.run(qc_merged_t,shots = 10000).result()
    counts = result.get_counts(qc_merged_t)
    #Showing top 10 bitsrings of the final state
    top_10_MPS = sorted(counts.items(), key=lambda item: item[1], reverse=True)[:2]
    for (k1, v1), (k2, v2) in zip(top_10_MPS, top_10_MPS):
                    print(f"{k1} = {v1:<16} |")
    print("\n")
    

Added deviation: 0.0
Mean block deviation: 0.0040099768334457666
000000 = 9898             |
010011 = 16               |


Added deviation: 0.01
Mean block deviation: 0.005812078118977153
000000 = 9895             |
010011 = 30               |


Added deviation: 0.02
Mean block deviation: 0.009252762001386272
000000 = 9833             |
010011 = 30               |


Added deviation: 0.03
Mean block deviation: 0.01318792861928108
000000 = 9732             |
000010 = 31               |


Added deviation: 0.04
Mean block deviation: 0.01624685387245758
000000 = 9635             |
000010 = 82               |


Added deviation: 0.05
Mean block deviation: 0.021021166404956784
000000 = 9504             |
000011 = 62               |


Added deviation: 0.06
Mean block deviation: 0.027393028029325178
000000 = 9080             |
000010 = 203              |


Added deviation: 0.07
Mean block deviation: 0.028635089858039364
000000 = 9154             |
000010 = 231              |


Added deviation: 0

In [9]:
n_q_list = [22, 24, 26]
n_l_list = [4, 8, 16, 32, 64, 128]
Relative_peak_probability_MPS = []
for i_q in range(9):
    for i_l in range(6):
        n_q = n_q_list[i_q]
        n_l = n_l_list[i_l]
        print(f"number of qubits: {n_q}")
        print(f"depth: {n_l}")
        prec = 0.005 # modification precision
        n_tries = 3 #number of tries to achieve modification precision
        phi_range = 2 * PI #range of angles that is used for U-gates in the random peaked quantum circuit (prior to modification)
        params_rand, mirror_params, mirror_params_init = peaked_circuit_parameters(n_q, n_l, prec, n_tries, phi_range)
        
        for i in range(15):
            added_deviation = 0.0 + (i)*0.01
            print(f"Added deviation: {added_deviation}")
            
            block_dev = []
            qc, block_dev = create_peaked_circuit(params_rand, mirror_params, mirror_params_init, n_q, n_l)
            #qc.draw('mpl')
            #show mean block deviation
            mean_block_deviation = np.mean(block_dev)
            print(f"Mean block deviation: {mean_block_deviation}")
            
            #simulate created peaked circuit
            simulator = AerSimulator()
            qc_merged_t = transpile(qc, simulator)
            result = simulator.run(qc_merged_t,shots = 10000).result()
            counts = result.get_counts(qc_merged_t)
            #Showing top 10 bitsrings of the final state
            top_10_MPS = sorted(counts.items(), key=lambda item: item[1], reverse=True)[:2]
            for (k1, v1), (k2, v2) in zip(top_10_MPS, top_10_MPS):
                            print(f"{k1} = {v1:<16} |")
            Relative_peak_probability_MPS.append(top_10_MPS[0][1]/top_10_MPS[1][1])
            print(f"Relative peak probability: {top_10_MPS[0][1]/top_10_MPS[1][1]}")
            print("\n")
    

number of qubits: 22
depth: 4
Added deviation: 0.0
Mean block deviation: 0.004728721067896728
0000000000000000000000 = 9060             |
1100000000000000000000 = 84               |
Relative peak probability: 107.85714285714286


Added deviation: 0.01
Mean block deviation: 0.006649265252464378
0000000000000000000000 = 8968             |
1100000000000000000000 = 107              |
Relative peak probability: 83.81308411214954


Added deviation: 0.02
Mean block deviation: 0.00997594297791045
0000000000000000000000 = 8923             |
1100000000000000000000 = 80               |
Relative peak probability: 111.5375


Added deviation: 0.03
Mean block deviation: 0.01383029963890729
0000000000000000000000 = 8605             |
0000000001000000000000 = 107              |
Relative peak probability: 80.42056074766356


Added deviation: 0.04
Mean block deviation: 0.017616531119807075
0000000000000000000000 = 8371             |
0011000000000000000000 = 86               |
Relative peak probability: 9

IndexError: list index out of range

In [8]:
len(Relative_peak_probability_MPS)

810